In [ ]:
from pathlib import Path

import warnings

warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (max_error, mean_absolute_error,
                             mean_absolute_percentage_error,
                             mean_squared_error, median_absolute_error,
                             r2_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")

# ch08 回归评估（练习版）

> 按提示补全 `____`，跑通所有 assert。数据与两个待评估模型已在脚手架就绪：
> `p_lin`（线性）与 `p_rf`（RF），`y_te` 是测试段真实值。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。

In [ ]:
# 数据与口径沿用 ch04：NUM 5 列 + OneHot(供电区域类型)，台区编号是 ID 列不进特征
reg = pd.read_csv(DATA / "load_reg.csv")
NUM = ["容量kVA", "投运年限", "月平均温度", "月平均湿度", "户数"]
CAT = ["供电区域类型"]
TARGET = "日均负荷"

tr_idx, te_idx = train_test_split(np.arange(len(reg)), test_size=0.2, random_state=42)
reg_tr, reg_te = reg.iloc[tr_idx], reg.iloc[te_idx]
y_tr = reg_tr[TARGET].to_numpy()
y_te = reg_te[TARGET].to_numpy()

oh = OneHotEncoder(sparse_output=False).fit(reg_tr[CAT])
Xtr = np.hstack([reg_tr[NUM].to_numpy(), oh.transform(reg_tr[CAT])])
Xte = np.hstack([reg_te[NUM].to_numpy(), oh.transform(reg_te[CAT])])
ss = StandardScaler().fit(Xtr)

# 两个待评估模型：线性（scaled）与 RF——本章重点是「怎么评」，不是「怎么训」
lin = LinearRegression().fit(ss.transform(Xtr), y_tr)
p_lin = lin.predict(ss.transform(Xte))
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, y_tr)
p_rf = rf.predict(Xte)

print("sklearn", sklearn.__version__, "| train/test:", len(tr_idx), "/", len(te_idx),
      "| y_te mean/std:", round(float(y_te.mean()), 4), round(float(y_te.std()), 4))
assert p_lin.shape == (160,) and p_rf.shape == (160,)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def reg_report(tag, y_true, y_pred):
    """一行输出 R2 / RMSE / MAE，返回 (r2, rmse, mae)。"""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{tag}: R2={r2:.4f} RMSE={rmse:.4f} MAE={mae:.4f}")
    return r2, rmse, mae


print("脚手架就绪：reg_report")

## 任务 1：均值基线

用训练段均值当预测，算出 MAE / RMSE / R²，看看「不作为」值多少分。

In [ ]:
# TODO(1)：造「均值基线」：用训练段均值当预测，算 MAE / RMSE / R²
# 提示：np.full(len(y_te), y_tr.mean())；三个指标各调一次 metrics 函数
base = ____
mae_b = ____
rmse_b = ____
r2_b = ____
print(f"均值基线: MAE={mae_b:.4f} RMSE={rmse_b:.4f} R2={r2_b:.4f}")
print("解读: R2 为负——比『用测试集均值当预测』还差一点，因为它用的是训练段均值")

assert round(mae_b, 4) == 139.8473
assert round(rmse_b, 4) == 175.6490
assert round(float(r2_b), 4) == -0.0123, "负 R²：模型上限就是『不作为』也它不如"

## 任务 2：手工复算 R²

R² 就是 1 - SSE/SST，手工算一遍与 sklearn 对照。

In [ ]:
# TODO(2)：手工复算 R²：1 - SSE/SST，与 sklearn 对照
# 提示：sse = ((y_te - p_lin) ** 2).sum()；sst = ((y_te - y_te.mean()) ** 2).sum()
sse = ____
sst = ____
r2_manual = ____
print("手工 R2:", round(float(r2_manual), 6), "| sklearn:",
      round(float(r2_score(y_te, p_lin)), 6))

assert round(float(r2_manual), 6) == round(float(r2_score(y_te, p_lin)), 6) == 0.923381
assert round(float(r2_manual), 4) == 0.9234

## 任务 3：R² 的三种极端

预测全取负、常数=测试集均值、常数=训练段均值，各算一次 R²。

In [ ]:
# TODO(3)：R² 的三种极端：预测全取负 / 常数=测试集均值 / 常数=训练段均值
# 提示：r2_score(y_te, -p_lin)；r2_score(y_te, np.full(len(y_te), y_te.mean()))
r2_neg = ____
r2_te_mean = ____
r2_tr_mean = ____
print(f"预测取负: R2={r2_neg:.4f} | 常数=测试集均值: {r2_te_mean:.4f} | 常数=训练段均值: {r2_tr_mean:.4f}")

assert round(float(r2_neg), 4) == -21.0964, "方向反了能有多惨，R² 没有下限"
assert round(float(r2_te_mean), 6) == 0.0, "R² 的 0 点定义就是测试集均值"
assert round(float(r2_tr_mean), 4) == -0.0123

## 任务 4：MAPE 的单位不变性

y 与预测同乘 0.1，验证 MAE 缩水、MAPE 不变；再用合成数据看小分母爆炸。

In [ ]:
# TODO(4)：MAPE 的单位不变性：y 与预测同乘 0.1，MAE 变 MAPE 不变
# 提示：mean_absolute_percentage_error(y_te * 0.1, p_lin * 0.1)
mape_full = ____
mape_scaled = ____
print(f"MAPE 原始={mape_full:.4f} | 同乘 0.1 后={mape_scaled:.6f}（不变）| MAE 变为 {mean_absolute_error(y_te * 0.1, p_lin * 0.1):.4f}")

# MAPE 的真陷阱：y 接近 0 时分母爆炸（合成演示，本数据 y 最小 103.64 安全）
y_fake = np.array([100.0, 100.0, 1.0])
p_fake = np.array([90.0, 110.0, 2.0])
print("合成小分母 MAPE:", round(float(mean_absolute_percentage_error(y_fake, p_fake)), 4),
      "← 第三行 100% 误差独占大头")
print("对照 MAE:", round(float(mean_absolute_error(y_fake, p_fake)), 4))

assert round(float(mape_full), 4) == 0.1203
assert round(float(mape_scaled), 6) == 0.120294
assert round(float(mean_absolute_percentage_error(y_fake, p_fake)), 4) == 0.4, "0.1+0.1+1.0 的均值"

## 任务 5：残差基本量

e = y - y_hat，算线性与 RF 的残差均值与线性残差的 std。

In [ ]:
# TODO(5)：残差基本量：e = y - y_hat，算线性与 RF 的残差均值
# 提示：res_lin = y_te - p_lin
res_lin = ____
res_rf = ____
mean_lin = ____
mean_rf = ____
print(f"线性残差 mean={mean_lin:.4f} std={float(res_lin.std()):.4f} | RF 残差 mean={mean_rf:.4f}")
print("解读: 残差均值≈0 是无偏的必要条件，但≠0 也不报错——要自己查")

assert round(mean_lin, 4) == 3.0419
assert round(mean_rf, 4) == 1.9413
assert round(float(res_lin.std()), 4) == 48.2273

## 任务 6：大误差定位

|残差| 的 p90，以及超过 p90 的样本真实负荷均值——误差藏在哪？

In [ ]:
# TODO(6)：大误差定位：|残差| 的 p90，以及超过 p90 的样本真实负荷均值
# 提示：np.quantile(np.abs(res_lin), 0.9)
q90 = ____
big = ____
big_mean = ____
print(f"|残差| p90={q90:.4f} | 大误差样本均值={big_mean:.4f} vs 全体 {float(y_te.mean()):.4f}")
print(f"RF 的 p90={float(np.quantile(np.abs(res_rf), 0.9)):.4f}（把大误差也压小了）")

assert round(q90, 4) == 79.0801
assert round(big_mean, 4) == 367.6875
assert round(float(np.quantile(np.abs(res_rf), 0.9)), 4) == 60.9175

## 任务 7：离群敏感对照

median_AE vs MAE，加 max_error——验证误差右偏。

In [ ]:
# TODO(7)：离群敏感对照：median_AE vs MAE，以及 max_error
# 提示：median_absolute_error(y_te, p_lin)；max_error(y_te, p_lin)
med_lin = ____
med_rf = ____
mx_lin = ____
mx_rf = ____
print(f"median_AE: 线性={med_lin:.4f} RF={med_rf:.4f} | MAE: 线性={mean_absolute_error(y_te, p_lin):.4f}")
print(f"max_error: 线性={mx_lin:.4f} RF={mx_rf:.4f}")
print("解读: median < mean 说明误差右偏（少数大离群）；报告里带一个 max_error 更诚实")

assert round(float(med_lin), 4) == 28.3693
assert round(float(med_rf), 4) == 23.3062
assert round(float(mx_lin), 4) == 175.2451
assert round(float(mx_rf), 4) == 115.9893

## 任务 8：交叉验证里的 neg_ 前缀

5 折交叉验证，scoring 分别用 r2 与 neg_root_mean_squared_error。

In [ ]:
# TODO(8)：交叉验证里的回归指标：scoring 用 neg_ 前缀，越接近 0 越好
# 提示：cross_val_score(model, X, y, cv=5, scoring="neg_root_mean_squared_error")
X_all = ____
cv_r2 = ____
cv_nrmse = ____
print("cv R2:", np.round(cv_r2, 4).tolist())
print("cv negRMSE:", np.round(cv_nrmse, 4).tolist(), "← 全是负的：sklearn 惯例『越大越好』")
print("真正的 RMSE =", np.round(-cv_nrmse, 4).tolist())

assert np.round(cv_r2, 4).tolist() == [0.9476, 0.9387, 0.9463, 0.9447, 0.9335]
assert np.round(cv_nrmse, 4).tolist() == [-41.9493, -46.0615, -44.3359, -45.0196, -47.8884]
assert (cv_nrmse < 0).all(), "neg_ 前缀指标恒为负"

## 完成标准

所有 assert 通过 = 任务完成。自查：R² 的 0 点定义、negRMSE 的符号、
median 与 MAE 的大小关系，三个概念都能脱稿说出。